# Wi-Fi CSI Presence Model Comparison — 20 MHz and 40 MHz

This is the **clean, ordered run notebook**. It compares Logistic Regression, Random Forest, and an Artificial Neural Network (ANN/MLP) on both bandwidth datasets in one run.

## Run order

Run **Cell 1 → Cell 2 → Cell 3 → Cell 4**, from top to bottom. Do not run the cells in your old research notebook again; this notebook performs just the combined model comparison.

- **Cell 1:** Connects Drive, locates `20MHz.zip`, and uses `40MHz.zip` from Drive or downloads it from Zenodo if absent.
- **Cell 2:** Extracts CSI features for both bandwidths and prints the dataset statistics and quality audit together.
- **Cell 3:** Evaluates all three models and prints one combined metrics table plus confusion matrices.
- **Cell 4:** Bundles the CSV tables and images into one ZIP and downloads it.

### Before running

Keep `20MHz.zip` in the root of Google Drive (`My Drive/20MHz.zip`). If you already have `40MHz.zip`, put it in the same location. Otherwise, Cell 1 attempts to fetch the official archive from the Zenodo record and verify its checksum. A large download may take several minutes.

### Validation caveat

The 20 MHz experiment uses leave-one-recording-date-out evaluation. The 40 MHz experiment uses leave-one-capture-out because the 40 MHz data has recording-date/class confounding. Results are exploratory and **not directly comparable as a bandwidth ranking**. Do not interpret simulation or recorded-data metrics as proof of live or through-wall sensing.

**Capture grouping fix:** Each MATLAB file is treated as one capture. If a class still has fewer than two independent files, the evaluation is marked `SKIPPED` and the audit is exported.


In [ ]:
# CELL 1 — SETUP, CONNECT DRIVE, AND PREPARE BOTH DATASETS
import os
import re
import gc
import json
import shutil
import hashlib
import zipfile
import warnings
import requests
from pathlib import Path, PurePosixPath

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive, files

from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
)

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
ZIP20 = DRIVE_ROOT / '20MHz.zip'
ZIP40_DRIVE = DRIVE_ROOT / '40MHz.zip'
ZIP40_RUNTIME = Path('/content/40MHz.zip')
OUT = Path('/content/wifi_csi_bandwidth_model_results')
TEMP = Path('/content/wifi_csi_current_capture.mat')
OUT.mkdir(parents=True, exist_ok=True)

if not ZIP20.is_file():
    candidates = list(DRIVE_ROOT.glob('**/20MHz.zip'))
    if candidates:
        ZIP20 = candidates[0]
    else:
        raise FileNotFoundError(
            'Could not find 20MHz.zip in Google Drive. Put it in My Drive root '
            'or update ZIP20 to its location.'
        )

ZENODO_API = 'https://zenodo.org/api/records/17225694'

def official_40mhz_info():
    response = requests.get(ZENODO_API, timeout=60)
    response.raise_for_status()
    record = response.json()
    info = next((item for item in record.get('files', []) if item.get('key') == '40MHz.zip'), None)
    if info is None:
        raise RuntimeError('40MHz.zip was not found in the official Zenodo record.')
    return info

def md5_file(path):
    digest = hashlib.md5()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

info40 = official_40mhz_info()
checksum_text = str(info40.get('checksum', ''))
expected_md5 = checksum_text.split(':', 1)[1].lower() if checksum_text.lower().startswith('md5:') else None

if ZIP40_DRIVE.is_file():
    ZIP40 = ZIP40_DRIVE
elif ZIP40_RUNTIME.is_file():
    ZIP40 = ZIP40_RUNTIME
else:
    ZIP40 = ZIP40_RUNTIME
    download_url = (info40.get('links', {}).get('content')
                    or info40.get('links', {}).get('download')
                    or 'https://zenodo.org/records/17225694/files/40MHz.zip?download=1')
    print('Downloading official 40MHz.zip from Zenodo. This can take several minutes...')
    with requests.get(download_url, stream=True, timeout=(30, 180),
                      headers={'User-Agent': 'WiFi-CSI-Student-Project/1.0'}) as response:
        response.raise_for_status()
        if 'json' in response.headers.get('Content-Type', '').lower():
            raise RuntimeError('Zenodo returned JSON instead of the ZIP archive.')
        with open(ZIP40, 'wb') as destination:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    destination.write(chunk)

print('20 MHz archive:', ZIP20, f'({ZIP20.stat().st_size / (1024**2):.1f} MiB)')
print('40 MHz archive:', ZIP40, f'({ZIP40.stat().st_size / (1024**2):.1f} MiB)')
if expected_md5:
    actual_md5 = md5_file(ZIP40)
    print('40 MHz official MD5:', expected_md5)
    print('40 MHz actual MD5:  ', actual_md5)
    if actual_md5 != expected_md5:
        raise RuntimeError('40MHz.zip checksum mismatch. Do not use this archive; re-download it.')
else:
    print('No MD5 checksum was exposed in the current Zenodo metadata; archive checksum was not compared.')

# Feature/preprocessing settings are fixed in advance.
WINDOW20 = 256
HOP20 = 128
WINDOW40 = 512
MIN_FINAL_WINDOW40 = 256
MIN_VALID_FRACTION40 = 0.90
RANDOM_STATE = 42

print('\nSettings:')
print(f'20 MHz: window={WINDOW20}, hop={HOP20}, features/subcarrier=4')
print(f'40 MHz: window={WINDOW40}, hop={WINDOW40}, min_length={MIN_FINAL_WINDOW40}, valid_fraction>={MIN_VALID_FRACTION40:.0%}')
print('Output directory:', OUT)

In [ ]:
# CELL 2 — EXTRACT FEATURES FOR BOTH BANDWIDTHS AND PRINT DATASET STATISTICS
def date_from_path(path_text, cfg=None):
    match = re.search(r'(?<!\d)(20\d{2}-\d{2}-\d{2})(?!\d)', str(path_text))
    if match:
        return match.group(1)
    if isinstance(cfg, dict):
        value = cfg.get('date')
        if isinstance(value, str) and re.fullmatch(r'20\d{2}-\d{2}-\d{2}', value):
            return value
    return 'unknown'


def people_count_from_path(path_text):
    match = re.search(r'(?:^|/)(\d+)people(?:/|$)', str(path_text))
    return int(match.group(1)) if match else None


def nearest_config(member, config_map_lower):
    parent = PurePosixPath(member).parent
    while str(parent) not in ('', '.'):
        candidate = str(parent / 'config.json').lower()
        if candidate in config_map_lower:
            return config_map_lower[candidate]
        parent = parent.parent
    return None


def read_csi_amplitude(mat_path, expected_subcarriers):
    with h5py.File(mat_path, 'r') as h5:
        if 'capture' not in h5:
            raise KeyError('MAT file does not contain a capture dataset')
        raw = h5['capture'][()]

    if raw.dtype.fields and {'real', 'imag'}.issubset(raw.dtype.fields):
        real = np.asarray(raw['real'], dtype=np.float32)
        imag = np.asarray(raw['imag'], dtype=np.float32)
    elif np.iscomplexobj(raw):
        real = np.asarray(raw.real, dtype=np.float32)
        imag = np.asarray(raw.imag, dtype=np.float32)
    else:
        raise TypeError(f'Unsupported CSI dtype: {raw.dtype}')
    del raw

    if real.ndim != 2 or imag.shape != real.shape:
        raise ValueError(f'Expected 2-D complex CSI, got {real.shape} and {imag.shape}')
    if real.shape[0] != expected_subcarriers and real.shape[1] == expected_subcarriers:
        real, imag = real.T, imag.T
    if real.shape[0] != expected_subcarriers:
        raise ValueError(f'Expected {expected_subcarriers} subcarriers; got shape {real.shape}')

    finite_columns = np.isfinite(real).all(axis=0) & np.isfinite(imag).all(axis=0)
    zero_columns = np.all((real == 0) & (imag == 0), axis=0)
    invalid_columns = zero_columns | (~finite_columns)
    real = np.nan_to_num(real, nan=0.0, posinf=0.0, neginf=0.0)
    imag = np.nan_to_num(imag, nan=0.0, posinf=0.0, neginf=0.0)
    amplitude = np.hypot(real, imag).astype(np.float32)
    del real, imag
    return amplitude, invalid_columns, int(zero_columns.sum())


def four_feature_groups(segment, valid_mask=None):
    # Returns four statistics per subcarrier in the established feature order.
    if valid_mask is None:
        A = segment
        mean_abs_diff = np.mean(np.abs(np.diff(segment, axis=1)), axis=1) if segment.shape[1] > 1 else np.zeros(segment.shape[0])
    else:
        valid_mask = np.asarray(valid_mask, dtype=bool)
        A = segment[:, valid_mask]
        if segment.shape[1] > 1:
            pair_valid = valid_mask[:-1] & valid_mask[1:]
            if np.any(pair_valid):
                left = segment[:, :-1][:, pair_valid]
                right = segment[:, 1:][:, pair_valid]
                mean_abs_diff = np.mean(np.abs(right - left), axis=1)
            else:
                mean_abs_diff = np.zeros(segment.shape[0], dtype=np.float32)
        else:
            mean_abs_diff = np.zeros(segment.shape[0], dtype=np.float32)
    if A.shape[1] == 0:
        raise ValueError('No valid columns remain in a feature window')
    return np.concatenate([
        np.mean(A, axis=1),
        np.std(A, axis=1),
        mean_abs_diff,
        np.percentile(A, 90, axis=1) - np.percentile(A, 10, axis=1)
    ]).astype(np.float32)


def extract_bandwidth_dataset(zip_path, bandwidth_mhz):
    expected_subcarriers = 256 if bandwidth_mhz == 20 else 512
    window_size = WINDOW20 if bandwidth_mhz == 20 else WINDOW40
    hop = HOP20 if bandwidth_mhz == 20 else WINDOW40

    feature_rows, feature_vectors, capture_rows, window_rows = [], [], [], []
    temp_mat = Path('/content/wifi_csi_current_capture.mat')

    with zipfile.ZipFile(zip_path, 'r') as archive:
        members = [name for name in archive.namelist()
                   if not name.endswith('/')
                   and not PurePosixPath(name).name.startswith('._')
                   and '__MACOSX' not in PurePosixPath(name).parts]
        config_members = [name for name in members if PurePosixPath(name).name.lower() == 'config.json']
        config_map_lower = {}
        for config_name in config_members:
            try:
                with archive.open(config_name) as stream:
                    config_map_lower[config_name.lower()] = json.load(stream)
            except Exception as exc:
                print('Skipping unreadable config:', config_name, repr(exc))

        mat_members = sorted(name for name in members if name.lower().endswith('.mat'))
        print(f'\n{bandwidth_mhz} MHz archive: {len(mat_members)} MAT members, {len(config_map_lower)} readable configs')

        processed = 0
        for num, mat_member in enumerate(mat_members, start=1):
            cfg = nearest_config(mat_member, config_map_lower)
            if not isinstance(cfg, dict):
                continue
            modulation = str(cfg.get('modulation', '')).lower()
            if modulation and modulation != 'ax':
                continue

            people = cfg.get('people', {}) if isinstance(cfg.get('people', {}), dict) else {}
            cfg_present = people.get('present')
            path_people_count = people_count_from_path(mat_member)
            cfg_people_count = people.get('num')
            people_count = path_people_count if path_people_count is not None else cfg_people_count

            if bandwidth_mhz == 20:
                if not isinstance(cfg_present, bool):
                    if people_count is None:
                        continue
                    label = int(int(people_count) > 0)
                    label_source = 'path_people_count_fallback'
                else:
                    label = int(cfg_present)
                    label_source = 'config_people_present'
            else:
                # Match the established 40 MHz workflow: infer presence from the people-count directory.
                if path_people_count is not None:
                    label = int(path_people_count > 0)
                    label_source = 'people_count_path'
                    if isinstance(cfg_present, bool) and int(cfg_present) != label:
                        print('WARNING: path/config label mismatch:', mat_member, label, int(cfg_present))
                elif isinstance(cfg_present, bool):
                    label = int(cfg_present)
                    label_source = 'config_people_present_fallback'
                elif people_count is not None:
                    label = int(int(people_count) > 0)
                    label_source = 'config_people_count_fallback'
                else:
                    continue

            date_group = date_from_path(mat_member, cfg)
            if date_group == 'unknown':
                print('Skipping capture with no parseable recording date:', mat_member)
                continue

            # Each MATLAB file is a capture in this dataset; grouping all files in a parent folder collapses class counts.
            capture_id = str(PurePosixPath(mat_member))
            try:
                with archive.open(mat_member, 'r') as source, open(temp_mat, 'wb') as target:
                    shutil.copyfileobj(source, target)
                amplitude, invalid_columns, all_zero_count = read_csi_amplitude(temp_mat, expected_subcarriers)
                n_packets = amplitude.shape[1]
                minimum_packets = MIN_FINAL_WINDOW40 if bandwidth_mhz == 40 else WINDOW20
                if n_packets < minimum_packets:
                    print('Skipping short capture:', mat_member, n_packets)
                    continue

                accepted_windows = 0
                rejected_windows = 0
                for window_index, start in enumerate(range(0, n_packets - window_size + 1, hop) if bandwidth_mhz == 20 else range(0, n_packets, hop)):
                    end = min(start + window_size, n_packets)
                    length = end - start
                    if bandwidth_mhz == 20 and length != window_size:
                        continue
                    if bandwidth_mhz == 40 and length < MIN_FINAL_WINDOW40:
                        window_rows.append({
                            'bandwidth_mhz': bandwidth_mhz, 'capture_id': capture_id,
                            'recording_date': date_group, 'presence_label': label,
                            'window_index': window_index, 'packet_start': start,
                            'packet_end_exclusive': end, 'window_length': length,
                            'valid_fraction': float((~invalid_columns[start:end]).mean()),
                            'accepted_for_features': False,
                            'rejection_reason': 'shorter_than_minimum_window'
                        })
                        rejected_windows += 1
                        continue

                    segment = amplitude[:, start:end]
                    if bandwidth_mhz == 20:
                        valid_fraction = float((~invalid_columns[start:end]).mean())
                        feature = four_feature_groups(segment)
                        accepted = True
                        rejection_reason = ''
                    else:
                        local_valid = ~invalid_columns[start:end]
                        valid_fraction = float(local_valid.mean())
                        accepted = valid_fraction >= MIN_VALID_FRACTION40
                        rejection_reason = '' if accepted else 'valid_fraction_below_threshold'
                        if accepted:
                            feature = four_feature_groups(segment, local_valid)
                        else:
                            feature = None

                    window_rows.append({
                        'bandwidth_mhz': bandwidth_mhz, 'capture_id': capture_id,
                        'recording_date': date_group, 'presence_label': label,
                        'window_index': window_index, 'packet_start': start,
                        'packet_end_exclusive': end, 'window_length': length,
                        'valid_fraction': valid_fraction,
                        'accepted_for_features': accepted,
                        'rejection_reason': rejection_reason
                    })
                    if not accepted:
                        rejected_windows += 1
                        continue
                    feature_vectors.append(feature)
                    feature_rows.append({
                        'bandwidth_mhz': bandwidth_mhz, 'capture_id': capture_id,
                        'recording_date': date_group, 'presence_label': label,
                        'window_index': window_index, 'valid_fraction': valid_fraction
                    })
                    accepted_windows += 1

                capture_rows.append({
                    'bandwidth_mhz': bandwidth_mhz,
                    'capture_id': capture_id,
                    'recording_date': date_group,
                    'presence_label': label,
                    'people_count': people_count,
                    'label_source': label_source,
                    'modulation': modulation,
                    'subcarriers': expected_subcarriers,
                    'packet_columns': n_packets,
                    'all_zero_columns': all_zero_count,
                    'invalid_columns_including_nonfinite': int(invalid_columns.sum()),
                    'invalid_column_percent': 100.0 * float(invalid_columns.mean()),
                    'accepted_windows': accepted_windows,
                    'rejected_windows': rejected_windows,
                })
                processed += 1
                print(f'[{processed}] {bandwidth_mhz} MHz | {PurePosixPath(mat_member).name} | date={date_group} | presence={label} | accepted_windows={accepted_windows} | invalid_columns={all_zero_count}')
                del amplitude, invalid_columns
                gc.collect()
            except Exception as exc:
                print('ERROR processing capture:', mat_member, repr(exc))
                raise
            finally:
                if temp_mat.exists():
                    temp_mat.unlink()

    if not feature_vectors:
        raise RuntimeError(f'No usable {bandwidth_mhz} MHz windows were generated. Review the archive paths and config files.')

    X = np.vstack(feature_vectors).astype(np.float32)
    win_meta = pd.DataFrame(feature_rows)
    cap_meta = pd.DataFrame(capture_rows)
    win_audit = pd.DataFrame(window_rows)

    # Fail visibly if the fixed 40 MHz quality rule drops every window from a capture.
    missing = cap_meta.loc[cap_meta['accepted_windows'] == 0, 'capture_id'].tolist()
    if bandwidth_mhz == 40 and missing:
        raise RuntimeError(
            'The fixed 90% valid-column rule rejected every window in these 40 MHz captures: '
            + '; '.join(missing)
            + '. Inspect the window audit before changing the threshold; do not tune it to maximize accuracy.'
        )

    y = win_meta['presence_label'].to_numpy(dtype=np.int32)
    dates = win_meta['recording_date'].astype(str).to_numpy()
    capture_ids = win_meta['capture_id'].astype(str).to_numpy()
    print(f'\n{bandwidth_mhz} MHz features: {X.shape}; evaluated captures: {cap_meta.shape[0]}')
    print('Capture counts by label:')
    print(cap_meta.groupby('presence_label')['capture_id'].nunique().to_string())
    print('Recording dates by label:')
    print(cap_meta.groupby(['recording_date', 'presence_label'])['capture_id'].nunique().unstack(fill_value=0).to_string())

    return {
        'bandwidth_mhz': bandwidth_mhz,
        'X': X,
        'y': y,
        'dates': dates,
        'capture_ids': capture_ids,
        'window_metadata': win_meta,
        'capture_metadata': cap_meta,
        'window_audit': win_audit,
    }

print('Feature extraction helpers loaded.')

# Extract each bandwidth separately. Feature widths differ (1,024 vs 2,048), so do not concatenate them.
data20 = extract_bandwidth_dataset(ZIP20, 20)
data40 = extract_bandwidth_dataset(ZIP40, 40)

capture_audit = pd.concat([data20['capture_metadata'], data40['capture_metadata']], ignore_index=True)
window_audit = pd.concat([data20['window_audit'], data40['window_audit']], ignore_index=True)
capture_audit_path = OUT / 'capture_quality_audit.csv'
window_audit_path = OUT / 'window_quality_audit.csv'
capture_audit.to_csv(capture_audit_path, index=False)
window_audit.to_csv(window_audit_path, index=False)

print('\n=== Capture quality audit ===')
display(capture_audit.groupby(['bandwidth_mhz', 'presence_label']).agg(
    captures=('capture_id', 'nunique'),
    total_packets=('packet_columns', 'sum'),
    total_all_zero_columns=('all_zero_columns', 'sum'),
    accepted_windows=('accepted_windows', 'sum'),
    rejected_windows=('rejected_windows', 'sum')
).reset_index())

print('\n=== Label / date audit ===')
display(capture_audit.groupby(['bandwidth_mhz', 'recording_date', 'presence_label']).size().rename('capture_count').reset_index())

In [ ]:
# CELL 3 — TRAIN AND EVALUATE LOGISTIC REGRESSION, RANDOM FOREST, AND ANN
# If a bandwidth lacks enough independent captures in both classes, record it as SKIPPED
# rather than crashing the whole run. Cell 4 can still export all completed results and audits.

def build_models():
    return {
        'Logistic Regression': make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE)
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=200, min_samples_leaf=2,
            class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1
        ),
        'ANN (MLP)': make_pipeline(
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(64, 32), activation='relu', solver='adam',
                alpha=0.001, batch_size=64, learning_rate_init=0.001,
                max_iter=300, early_stopping=False, n_iter_no_change=25,
                random_state=RANDOM_STATE, tol=1e-4
            )
        )
    }


def balance_ann_training(X_train, y_train, random_state):
    rng = np.random.default_rng(random_state)
    classes, counts = np.unique(y_train, return_counts=True)
    if len(classes) != 2:
        raise ValueError(f'Training fold must contain both labels; found {classes.tolist()}')
    target_count = int(counts.max())
    all_indices = []
    for class_label in classes:
        class_indices = np.flatnonzero(y_train == class_label)
        selected = rng.choice(class_indices, size=target_count, replace=(len(class_indices) < target_count))
        all_indices.extend(selected.tolist())
    all_indices = np.asarray(all_indices, dtype=int)
    rng.shuffle(all_indices)
    return X_train[all_indices], y_train[all_indices]


def skipped_rows_for_bandwidth(data, protocol, reason):
    metadata = data['capture_metadata'].drop_duplicates('capture_id')
    counts = metadata['presence_label'].value_counts().to_dict()
    rows = []
    for model_name in build_models():
        rows.append({
            'bandwidth_mhz': data['bandwidth_mhz'],
            'model': model_name,
            'validation_protocol': protocol,
            'evaluation_status': 'SKIPPED',
            'skip_reason': reason,
            'capture_count': int(metadata['capture_id'].nunique()),
            'class_0_capture_count': int(counts.get(0, 0)),
            'class_1_capture_count': int(counts.get(1, 0)),
            'window_count': int(len(data['y'])),
            'window_accuracy': np.nan,
            'capture_accuracy': np.nan,
            'capture_balanced_accuracy': np.nan,
            'capture_precision_presence': np.nan,
            'capture_recall_presence': np.nan,
            'capture_f1_presence': np.nan,
            'capture_macro_f1': np.nan,
            'true_negative': np.nan, 'false_positive': np.nan,
            'false_negative': np.nan, 'true_positive': np.nan,
            'date_count': int(metadata['recording_date'].nunique()),
            'research_caveat': reason
        })
    print(f"\nSKIPPING {data['bandwidth_mhz']} MHz evaluation: {reason}")
    print('Class counts by independent MATLAB capture:', counts)
    return rows, [], [], []


def run_bandwidth_evaluation(data):
    bandwidth = data['bandwidth_mhz']
    X, y = data['X'], data['y']
    dates, capture_ids = data['dates'], data['capture_ids']
    capture_metadata = data['capture_metadata'].copy()

    if capture_metadata.groupby('capture_id')['presence_label'].nunique().max() != 1:
        raise ValueError(f'{bandwidth} MHz: inconsistent labels within a capture.')

    capture_level = capture_metadata.drop_duplicates('capture_id')
    label_counts = capture_level['presence_label'].value_counts()
    protocol = ('leave-one-recording-date-out; majority vote over windows'
                if bandwidth == 20 else
                'leave-one-MAT-capture-out; mean window probability per capture; exploratory')

    # Both classes need at least two independent captures for leave-one-capture-out testing.
    if len(label_counts) != 2 or int(label_counts.min()) < 2:
        counts = {int(k): int(v) for k, v in label_counts.to_dict().items()}
        reason = (
            f'Not enough independent captures per class for binary cross-validation. '
            f'Found label counts {counts}; each class needs at least two MATLAB captures. '
            f'Check Cell 2 capture IDs and the 40 MHz label audit.'
        )
        return skipped_rows_for_bandwidth(data, protocol, reason)

    if bandwidth == 20:
        split_groups = dates
        for date in np.unique(dates):
            if len(np.unique(y[dates == date])) != 2:
                reason = f'At least one 20 MHz held-out date ({date}) does not contain both classes.'
                return skipped_rows_for_bandwidth(data, protocol, reason)
    else:
        split_groups = capture_ids
        date_class = capture_level.groupby('recording_date')['presence_label'].agg(['min', 'max'])
        if not (date_class['min'] != date_class['max']).any():
            print('CAUTION 40 MHz: class labels are associated with recording date. '
                  'Capture-held-out scores are exploratory and may reflect date/environment differences.')

    splitter = LeaveOneGroupOut()
    splits = list(splitter.split(X, y, groups=split_groups))
    model_templates = build_models()
    summary_rows, pred_rows, fold_rows, figure_paths = [], [], [], []

    for model_name, template in model_templates.items():
        print(f'\n===== {bandwidth} MHz | {model_name} =====')
        window_pred = np.full(len(y), -1, dtype=np.int32)
        window_prob = np.full(len(y), np.nan, dtype=np.float64)

        for fold_no, (train_idx, test_idx) in enumerate(splits, start=1):
            train_labels = np.unique(y[train_idx])
            if len(train_labels) != 2:
                raise ValueError(
                    f'{bandwidth} MHz fold {fold_no}: training labels are {train_labels.tolist()}. '
                    'There are not enough independent captures in both classes.'
                )
            if len(test_idx) == 0:
                raise ValueError(f'{bandwidth} MHz fold {fold_no}: empty test set')

            model = clone(template)
            X_train, y_train = X[train_idx], y[train_idx]
            before_count = len(y_train)
            if model_name == 'ANN (MLP)':
                X_fit, y_fit = balance_ann_training(
                    X_train, y_train, RANDOM_STATE + fold_no + bandwidth
                )
            else:
                X_fit, y_fit = X_train, y_train

            group_values = np.unique(split_groups[test_idx]).astype(str).tolist()
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always')
                model.fit(X_fit, y_fit)
                classes = model.classes_ if hasattr(model, 'classes_') else model.named_steps[
                    'logisticregression' if model_name == 'Logistic Regression' else 'mlpclassifier'
                ].classes_
                # The class-1 column is selected explicitly, not assumed to be column index 1.
                probability_columns = model.predict_proba(X[test_idx])
                positive_col = list(classes).index(1)
                probs = probability_columns[:, positive_col]
                preds = (probs >= 0.5).astype(np.int32)

            warning_messages = [str(w.message) for w in caught]
            window_pred[test_idx] = preds
            window_prob[test_idx] = probs
            ann_iterations = np.nan
            if model_name == 'ANN (MLP)':
                ann_iterations = int(model.named_steps['mlpclassifier'].n_iter_)

            fold_rows.append({
                'bandwidth_mhz': bandwidth, 'model': model_name, 'fold': fold_no,
                'validation_protocol': protocol,
                'held_out_group': ','.join(group_values),
                'train_windows_before_ANN_balancing': before_count,
                'train_windows_used': int(len(y_fit)), 'test_windows': int(len(test_idx)),
                'ann_iterations': ann_iterations,
                'convergence_warning': any('ConvergenceWarning' in type(w.message).__name__ for w in caught),
                'warnings': ' | '.join(warning_messages[:3])
            })
            print(f'Fold {fold_no}/{len(splits)} held out {group_values}; tested {len(test_idx)} windows')

        if np.any(window_pred < 0) or not np.isfinite(window_prob).all():
            raise RuntimeError(f'{bandwidth} MHz {model_name}: one or more windows lack an out-of-fold prediction.')

        true_capture, pred_capture = [], []
        for capture in capture_metadata['capture_id'].astype(str).unique():
            idx = np.flatnonzero(capture_ids == capture)
            if len(idx) == 0:
                continue
            actuals = np.unique(y[idx])
            if len(actuals) != 1:
                raise RuntimeError(f'Capture has inconsistent window labels: {capture}')
            actual = int(actuals[0])
            mean_prob = float(window_prob[idx].mean())
            if bandwidth == 20:
                vals, counts = np.unique(window_pred[idx], return_counts=True)
                predicted = int(vals[np.argmax(counts)])
                aggregate_rule = 'majority vote over window class predictions'
            else:
                predicted = int(mean_prob >= 0.5)
                aggregate_rule = 'mean probability >= 0.50'
            true_capture.append(actual)
            pred_capture.append(predicted)
            date_values = np.unique(dates[idx])
            pred_rows.append({
                'bandwidth_mhz': bandwidth, 'model': model_name,
                'validation_protocol': protocol, 'capture_id': capture,
                'recording_date': str(date_values[0]),
                'actual_presence': actual, 'predicted_presence': predicted,
                'mean_presence_probability': mean_prob,
                'correct': int(actual == predicted), 'window_count': int(len(idx)),
                'capture_aggregation_rule': aggregate_rule
            })

        cm = confusion_matrix(true_capture, pred_capture, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        summary_rows.append({
            'bandwidth_mhz': bandwidth,
            'model': model_name,
            'validation_protocol': protocol,
            'evaluation_status': 'COMPLETED',
            'skip_reason': '',
            'capture_count': int(len(true_capture)),
            'class_0_capture_count': int((capture_level['presence_label'] == 0).sum()),
            'class_1_capture_count': int((capture_level['presence_label'] == 1).sum()),
            'window_count': int(len(y)),
            'window_accuracy': float(accuracy_score(y, window_pred)),
            'capture_accuracy': float(accuracy_score(true_capture, pred_capture)),
            'capture_balanced_accuracy': float(balanced_accuracy_score(true_capture, pred_capture)),
            'capture_precision_presence': float(precision_score(true_capture, pred_capture, zero_division=0)),
            'capture_recall_presence': float(recall_score(true_capture, pred_capture, zero_division=0)),
            'capture_f1_presence': float(f1_score(true_capture, pred_capture, zero_division=0)),
            'capture_macro_f1': float(f1_score(true_capture, pred_capture, average='macro', zero_division=0)),
            'true_negative': int(tn), 'false_positive': int(fp),
            'false_negative': int(fn), 'true_positive': int(tp),
            'date_count': int(capture_metadata['recording_date'].nunique()),
            'research_caveat': ('Only two recording dates; exploratory' if bandwidth == 20 else
                                'Capture-held-out; date/class confounding possible; exploratory')
        })

        disp = ConfusionMatrixDisplay(cm, display_labels=['No person', 'Person present'])
        disp.plot(values_format='d')
        plt.title(f'{bandwidth} MHz — {model_name}\nCapture-level out-of-fold predictions')
        plt.tight_layout()
        safe_model = re.sub(r'[^a-z0-9]+', '_', model_name.lower()).strip('_')
        fig_path = OUT / f'confusion_matrix_{bandwidth}mhz_{safe_model}.png'
        plt.savefig(fig_path, dpi=160, bbox_inches='tight')
        plt.show()
        plt.close()
        figure_paths.append(fig_path)
        print(classification_report(
            true_capture, pred_capture, labels=[0, 1],
            target_names=['No person', 'Person present'], zero_division=0
        ))

    return summary_rows, pred_rows, fold_rows, figure_paths

# Process each bandwidth independently. A problem in 40 MHz will not erase valid 20 MHz outputs.
summary_rows, prediction_rows, fold_rows, confusion_images = [], [], [], []
for dataset in [data20, data40]:
    s, p, f, figs = run_bandwidth_evaluation(dataset)
    summary_rows.extend(s)
    prediction_rows.extend(p)
    fold_rows.extend(f)
    confusion_images.extend(figs)

summary_df = pd.DataFrame(summary_rows)
predictions_df = pd.DataFrame(prediction_rows)
fold_details_df = pd.DataFrame(fold_rows)

summary_path = OUT / 'bandwidth_model_comparison.csv'
predictions_path = OUT / 'bandwidth_capture_predictions.csv'
folds_path = OUT / 'bandwidth_fold_details.csv'
summary_df.to_csv(summary_path, index=False)
predictions_df.to_csv(predictions_path, index=False)
fold_details_df.to_csv(folds_path, index=False)

print('\n=== Combined model comparison / evaluation status ===')
cols = [
    'bandwidth_mhz', 'model', 'evaluation_status', 'validation_protocol',
    'capture_count', 'class_0_capture_count', 'class_1_capture_count',
    'capture_accuracy', 'capture_balanced_accuracy', 'capture_precision_presence',
    'capture_recall_presence', 'capture_f1_presence', 'capture_macro_f1',
    'false_positive', 'false_negative', 'skip_reason', 'research_caveat'
]
display(summary_df[[c for c in cols if c in summary_df.columns]].round(4))
print('\nSaved result files to:', OUT)


In [ ]:
# CELL 4 — PACKAGE ALL AVAILABLE RESULTS AND DOWNLOAD ONE ZIP
# This cell is defensive: it still works if one bandwidth was skipped by Cell 3.
zip_path = Path('/content/wifi_csi_bandwidth_model_results.zip')
expected_paths = [
    OUT / 'bandwidth_model_comparison.csv',
    OUT / 'bandwidth_capture_predictions.csv',
    OUT / 'bandwidth_fold_details.csv',
    capture_audit_path,
    window_audit_path,
]
# Include all newly generated model figures and any files present from the current run.
files_to_bundle = [p for p in expected_paths if Path(p).exists()]
files_to_bundle += sorted(OUT.glob('confusion_matrix_*mhz_*.png'))

if not files_to_bundle:
    raise FileNotFoundError('No result files were found. Run Cell 2 and Cell 3 before Cell 4.')

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files_to_bundle:
        archive.write(path, arcname=f'results/{Path(path).name}')

print('Created:', zip_path)
print('Files inside bundle:')
with zipfile.ZipFile(zip_path) as archive:
    for name in archive.namelist():
        print(' -', name)

files.download(str(zip_path))


## Files you will receive

- `bandwidth_model_comparison.csv` — one row per bandwidth/model with capture-level metrics and validation caveats.
- `bandwidth_capture_predictions.csv` — out-of-fold capture predictions.
- `bandwidth_fold_details.csv` — held-out groups, fold sizes, ANN iteration counts and warnings.
- `capture_quality_audit.csv` — capture-level data quality and accepted/rejected window counts.
- `window_quality_audit.csv` — per-window quality decisions.
- `confusion_matrix_*.png` — capture-level confusion matrices for all six bandwidth/model combinations.

When Colab downloads `wifi_csi_bandwidth_model_results.zip`, upload that ZIP back to the chat so the actual results can be reviewed before writing them into GitHub.
